# Bonsai Image Demo — Google Colab

Run the demo end-to-end on a Colab GPU runtime: clone, set up, launch the studio, open the UI through Colab's session proxy.

On Linux the demo uses the **ternary-gemlite** model (gemlite int2 kernels + HQQ-int4 TE + bf16 VAE via `backend_gpu`).

**Before you run anything: Runtime → Change runtime type → T4 GPU (or better).** Without a CUDA GPU, generation can't run.


## 0. Tokens

On the left-hand side click on secrets tab, and add `GH_TOKEN` and `HF_TOKEN`. This is only needed before we launch the model and demo repositories public.

In [ ]:
from google.colab import userdata
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    HF_TOKEN = userdata.get('HF_TOKEN')
    import os, subprocess
    assert GH_TOKEN and HF_TOKEN, 'set GH_TOKEN and HF_TOKEN above before running this cell.'
    subprocess.run(
        ['git', 'config', '--global',
        f'url.https://{GH_TOKEN}@github.com/.insteadOf',
        'https://github.com/'],
        check=True,
    )
    os.environ['BONSAI_TOKEN'] = HF_TOKEN
    del GH_TOKEN, HF_TOKEN  # don't leave them in the kernel namespace
    print('credentials wired into git + BONSAI_TOKEN.')

except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("Error: SecretNotFoundError or NotebookAccessError")


## 1. Sanity-check the runtime

Should print the NVIDIA GPU info. If you see *command not found* or *No devices were found*, switch to a GPU runtime first.


In [ ]:
!nvidia-smi


## 2. Clone the demo repo


In [ ]:
import os, shutil, subprocess

REPO = '/content/Bonsai-Image-Demo'
URL  = 'https://github.com/PrismML-Eng/Bonsai-Image-Demo.git'

# /content is ephemeral in Colab (wiped per session). Detect a half-cloned
# state by the absence of setup.sh — a present-but-empty dir is the common
# failure mode of an interrupted clone. Wipe + re-clone so the next steps
# can rely on the repo being intact.
if not os.path.isfile(f'{REPO}/setup.sh'):
    if os.path.exists(REPO):
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth=1', URL, REPO], check=True)
os.chdir(REPO)
print(f'cwd = {os.getcwd()}')


## 3. Run `setup.sh`

Installs `uv`, creates a Python venv, clones the vendor deps into `vendor/`, syncs Python + Node packages, and install dependencies.

`SKIP_DOWNLOAD=1` skips the auto model download here; the model gets pulled in its own cell next.


In [ ]:
!SKIP_DOWNLOAD=1 ./setup.sh


## 4. Download the model

Downloads `ternary-gemlite` variant which is used for gpu with gemlite kernels.
Pulls `prism-ml/bonsai-image-ternary-4B-gemlite-2bit` into `models/bonsai-image-4B-ternary-gemlite/`.

In [ ]:
!./scripts/download_model.sh ternary
!./scripts/download_model.sh binary


## 5. Launch the studio daemon (`serve.sh`)

- In the background, starts the frontend and backend servers for the bonsai image generation next.js app.
- Next cell, waits until both servers are ready. Backend, takes few mins due to warming up different image-gen shapes and kernel JIT compilation.


In [ ]:
import os, subprocess, urllib.request

REPO = '/content/Bonsai-Image-Demo'

def _backend_alive(port=8000):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{port}/healthz', timeout=1)
        return True
    except Exception:
        return False

if _backend_alive():
    print('serve.sh is already running on :8000 — reusing.')
else:
    import subprocess

    try:
        gpu_name = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            timeout=2,
        ).decode().strip().splitlines()[0]
    except Exception:
        gpu_name = "unknown"
    print(f"Detected GPU: {gpu_name}")


    log_dir = f'{REPO}/.serve-logs'
    os.makedirs(log_dir, exist_ok=True)
    log = open(f'{log_dir}/notebook.log', 'a')
    env = os.environ.copy()
    # Pre-warm every resolution offered by the studio's picker (5 aspects ×
    # 2 quality tiers). Each shape costs ~10-30s of one-time JIT/autotune
    # depending on size; total adds 3-5 min to boot but every first user
    # gen lands warm. Configs persist to outputs/.{gemlite,triton}_cache/.
    # All 10 picker shapes. Klein's diffusion_klein.py requires height and
    # width to be multiples of 32 (vae_scale_factor * 2); the fast-tier 3:2
    # entries are 576x384 / 384x576 (exact 3:2, both mult-of-32).
    if "T4" in gpu_name:
        # T4 is slow at JIT — limit to one shape so the readiness wait
        env.setdefault('BONSAI_WARMUP_SHAPES', '512x512')

        # increase default timeout (specially needed on T4 instances)
        env.setdefault('BACKEND_READY_TIMEOUT', '1800')
    else:
        env.setdefault(
            'BONSAI_WARMUP_SHAPES',
            '512x512,1024x1024,'
            '576x384,1248x832,'
            '384x576,832x1248,'
            '704x352,1408x704,'
            '352x704,704x1408',
        )
        env.setdefault('BACKEND_READY_TIMEOUT', '900')

    # Production frontend build (no HMR WebSocket): Colab's session proxy
    # doesn't tunnel WS, and the dev client crashes hydration on repeated
    # connect failures — buttons end up inert. `next build` once + `next
    # start` sidesteps it entirely. First boot adds ~60s for the build.
    env.setdefault('BONSAI_FRONTEND_PROD', '1')
    # Absolute path for the executable: subprocess.Popen resolves argv[0]
    # relative to the *parent* cwd before applying cwd=.
    proc = subprocess.Popen(
        [f'{REPO}/scripts/serve.sh'],
        cwd=REPO, stdout=log, stderr=subprocess.STDOUT, env=env,
        start_new_session=True,  # detach from notebook signals
    )
    print(f'serve.sh started, PID={proc.pid}')
    print(f'logs: {log_dir}/notebook.log')
    print(f'while waiting, in a fresh cell:  !tail -n 50 {log_dir}/notebook.log')


In [ ]:
# Blocks until both endpoints respond at the HTTP layer. Uses HTTP, not
# socket.connect — uvicorn binds the TCP listener *before* the lifespan
# completes prewarm, so a raw socket connect returns true too early and
# the frontend's first /api/generate call fails with 'Could not reach
# backend'. /healthz is only served after lifespan startup finishes.
import time, urllib.request

def _http_ok(url, timeout=1):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status < 500
    except Exception:
        return False

checks = {
    'backend':  'http://127.0.0.1:8000/healthz',
    'frontend': 'http://127.0.0.1:3000/',
}

# Backend startup = model load + one diffusion forward per shape in
# BONSAI_WARMUP_SHAPES — total scales with how many shapes you warm
# (~tens of seconds per shape). Frontend binds within a couple seconds.
# \r overwrites the same line each poll so the output stays a single
# row instead of piling up timestamps.
t0 = time.time()
while time.time() - t0 < 360:
    state = {name: _http_ok(u) for name, u in checks.items()}
    msg = '  '.join(f"{name}: {'up' if v else 'starting'}" for name, v in state.items())
    print(f'\r[{int(time.time() - t0):3d}s] {msg}    ', end='', flush=True)
    if all(state.values()):
        break
    time.sleep(2)

final = {name: _http_ok(u) for name, u in checks.items()}
elapsed = time.time() - t0
if all(final.values()):
    print(f'\nready in {elapsed:.1f}s')
else:
    print(f'\nFAILED after {elapsed:.1f}s: {final}')
    raise RuntimeError('one or more endpoints never responded — tail .serve-logs/notebook.log')


## 6. Open the studio

Embeds the proxied `:3000` page as an iframe in this cell. To fullscreen it, click the **three-dot menu** on the left side of the cell and choose **View cell output in fullscreen**.


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(3000, height='1600', width='100%')


## 7. Stop the daemon

The kernel owns `serve.sh`; resetting/disconnecting the Colab runtime kills it. To stop it explicitly without restarting the kernel, kill anything bound to `:8000` and `:3000`.


In [ ]:
import subprocess

def stop_daemon(ports=(8000, 3000)):
    """Kill listeners on the given ports. Call manually — `Run all` skips this."""
    for port in ports:
        pids = subprocess.run(
            ['lsof', '-nP', '-iTCP:' + str(port), '-sTCP:LISTEN', '-t'],
            capture_output=True, text=True,
        ).stdout.split()
        for pid in pids:
            subprocess.run(['kill', '-TERM', pid])
            print(f'killed PID {pid} on :{port}')

# Uncomment to stop the daemon without restarting the kernel:
# stop_daemon()